In [2]:
"""What this script does:
  1. Loads cleaned data from SQLite database (built in Part A)
  2. Splits data into 80% training and 20% test sets
  3. Addresses class imbalance using class_weight='balanced'
  4. Trains two classification algorithms:
     - Logistic Regression (simple, interpretable baseline)
     - Random Forest (ensemble, handles non-linear patterns)
  5. Evaluates both models using:
     - Accuracy, Precision, Recall, F1 Score
     - Confusion Matrix (visualised as heatmap)
  6. Compares both models and states which performed better and why
 
WHY THESE TWO ALGORITHMS:
  Logistic Regression — finds the best linear boundary between classes.
  Simple, fast, and interpretable. Used as a baseline.
  If a complex model can't beat this, something is wrong.
 
  Random Forest — builds 100 decision trees on random subsets of data,
  then takes a majority vote. Handles non-linear patterns, resistant to
  overfitting, and provides feature importance scores.
"""


import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, confusion_matrix, classification_report
)
 
sns.set_theme(style="whitegrid")
os.makedirs("charts", exist_ok=True)

In [3]:
# STEP 1: Load cleaned data from SQLite
# ─────────────────────────────────────────────
# We always load from the database — one source of truth.
# No re-cleaning, no re-processing needed.
 
print("=" * 55)
print("STEP 1: Loading data from heart_data.db")
print("=" * 55)
 
conn = sqlite3.connect("heart_data.db")
df = pd.read_sql_query("SELECT * FROM patients", conn)
conn.close()
 
print(f"Loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Class distribution:\n{df['heart_disease'].value_counts()}")
print(f"\n0 = No Disease | 1 = Has Disease")

STEP 1: Loading data from heart_data.db
Loaded: 208 rows × 14 columns
Class distribution:
0    127
1     81
Name: heart_disease, dtype: int64

0 = No Disease | 1 = Has Disease


In [4]:
# STEP 2: Define features and target
# ─────────────────────────────────────────────
# X = all columns except the target (the features the model learns from)
# y = the target column (what the model is trying to predict)
 
print("\n" + "=" * 55)
print("STEP 2: Defining features (X) and target (y)")
print("=" * 55)
 
X = df.drop(columns=["heart_disease"])
y = df["heart_disease"]
 
print(f"Features (X): {X.shape[1]} columns")
print(f"Target  (y): '{y.name}' — values: {y.unique()}")


STEP 2: Defining features (X) and target (y)
Features (X): 13 columns
Target  (y): 'heart_disease' — values: [0 1]


In [5]:
# STEP 3: Train/Test Split — 80/20
# ─────────────────────────────────────────────
# test_size=0.2    → 20% of data goes to test set (44 patients)
# random_state=42  → fixes the random shuffle so results are reproducible
#                    (any number works — 42 is a convention)
# stratify=y       → ensures both train and test sets have the same
#                    class ratio as the full dataset (61/39)
#                    Without this, by bad luck test might be all one class
 
print("\n" + "=" * 55)
print("STEP 3: Train / Test Split (80/20)")
print("=" * 55)
 
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y          # preserves class ratio in both splits
)
 
print(f"Training set : {X_train.shape[0]} patients")
print(f"Test set     : {X_test.shape[0]} patients")
print(f"\nClass distribution in training set:")
print(y_train.value_counts())
print(f"\nClass distribution in test set:")
print(y_test.value_counts())


STEP 3: Train / Test Split (80/20)
Training set : 166 patients
Test set     : 42 patients

Class distribution in training set:
0    101
1     65
Name: heart_disease, dtype: int64

Class distribution in test set:
0    26
1    16
Name: heart_disease, dtype: int64


In [6]:
# STEP 4: Feature Scaling
# ─────────────────────────────────────────────
# Logistic Regression is sensitive to feature scale.
# For example: age ranges 29–76, cholesterol ranges 126–564.
# Without scaling, cholesterol would dominate simply because its numbers
# are bigger — not because it's more important.
#
# StandardScaler transforms each feature to have mean=0, std=1.
# This puts all features on the same scale.
#
# IMPORTANT: We fit the scaler ONLY on training data, then apply it to test.
# If we fit on the full dataset, test data information leaks into training —
# that's called data leakage and it inflates your results artificially.
 
print("\n" + "=" * 55)
print("STEP 4: Feature Scaling (StandardScaler)")
print("=" * 55)
 
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # learn scale from train only
X_test_scaled  = scaler.transform(X_test)         # apply same scale to test
 
print("Scaling applied — all features now have mean≈0 and std≈1")
print("Scaler fitted on training data only (no data leakage)")


STEP 4: Feature Scaling (StandardScaler)
Scaling applied — all features now have mean≈0 and std≈1
Scaler fitted on training data only (no data leakage)


In [7]:
# STEP 5: Address Class Imbalance
# ─────────────────────────────────────────────
# From Part B: 127 no disease (61%) vs 81 disease (39%)
# This is a mild but real imbalance.
#
# APPROACH: class_weight='balanced'
# This tells the model: "Penalise mistakes on the minority class more heavily."
# Mathematically, it multiplies the loss for minority class errors by
# (total_samples / (2 * minority_class_count))
#
# WHY NOT SMOTE HERE?
# SMOTE synthetically creates new minority class samples.
# class_weight='balanced' is simpler, requires no new data generation,
# and is less prone to introducing artificial patterns.
# For a mild imbalance like ours (61/39), class_weight is sufficient.
 
print("\n" + "=" * 55)
print("STEP 5: Addressing Class Imbalance")
print("=" * 55)
print("Method: class_weight='balanced' in both models")
print("Effect: minority class (disease) errors are penalised more heavily")
print("Reason: mild 61/39 imbalance — class weighting is sufficient")
print("        SMOTE would be preferred for severe imbalances (e.g. 95/5)")
 


STEP 5: Addressing Class Imbalance
Method: class_weight='balanced' in both models
Effect: minority class (disease) errors are penalised more heavily
Reason: mild 61/39 imbalance — class weighting is sufficient
        SMOTE would be preferred for severe imbalances (e.g. 95/5)


In [8]:
# STEP 6: Train Model 1 — Logistic Regression
# ─────────────────────────────────────────────
# max_iter=1000 gives the algorithm enough iterations to converge
# (default 100 sometimes throws a convergence warning)
 
print("\n" + "=" * 55)
print("STEP 6: Training Model 1 — Logistic Regression")
print("=" * 55)
 
lr_model = LogisticRegression(
    class_weight="balanced",
    random_state=42,
    max_iter=1000
)
lr_model.fit(X_train_scaled, y_train)
lr_predictions = lr_model.predict(X_test_scaled)
 
print("Logistic Regression trained successfully.")


STEP 6: Training Model 1 — Logistic Regression
Logistic Regression trained successfully.


In [9]:
# STEP 7: Train Model 2 — Random Forest
# ─────────────────────────────────────────────
# n_estimators=100 → build 100 decision trees and take majority vote
# Random Forest does NOT need feature scaling — it uses decision boundaries
# not distance calculations, so feature magnitude doesn't matter
 
print("\n" + "=" * 55)
print("STEP 7: Training Model 2 — Random Forest")
print("=" * 55)
 
rf_model = RandomForestClassifier(
    n_estimators=100,          # 100 decision trees
    class_weight="balanced",
    random_state=42
)
rf_model.fit(X_train, y_train)          # uses unscaled data — RF doesn't need scaling
rf_predictions = rf_model.predict(X_test)
 
print("Random Forest trained successfully (100 trees).")


STEP 7: Training Model 2 — Random Forest
Random Forest trained successfully (100 trees).


In [10]:
# STEP 8: Evaluate Both Models
# ─────────────────────────────────────────────
# We evaluate on the TEST set only — data the model has never seen.
#
# Accuracy  = (TP + TN) / Total — overall correct predictions
# Precision = TP / (TP + FP)   — of all flagged as sick, how many were right
# Recall    = TP / (TP + FN)   — of all actually sick, how many were caught
# F1        = 2 * (Precision * Recall) / (Precision + Recall) — balance of both
#
# In medical screening: RECALL is the most critical metric
# Missing a sick patient (False Negative) is more dangerous than
# a false alarm (False Positive)
 
print("\n" + "=" * 55)
print("STEP 8: Model Evaluation")
print("=" * 55)
 
def evaluate_model(name, y_true, y_pred):
    """Calculate and print all evaluation metrics for a model."""
    acc  = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred, zero_division=0)
    rec  = recall_score(y_true, y_pred, zero_division=0)
    f1   = f1_score(y_true, y_pred, zero_division=0)
 
    print(f"\n── {name} ──")
    print(f"  Accuracy  : {acc:.4f}  ({acc*100:.1f}%)")
    print(f"  Precision : {prec:.4f}")
    print(f"  Recall    : {rec:.4f}  ← most important in medical context")
    print(f"  F1 Score  : {f1:.4f}")
    print(f"\n  Classification Report:")
    print(classification_report(y_true, y_pred,
          target_names=["No Disease", "Has Disease"]))
 
    return {"name": name, "accuracy": acc, "precision": prec,
            "recall": rec, "f1": f1}
 
lr_metrics = evaluate_model("Logistic Regression", y_test, lr_predictions)
rf_metrics = evaluate_model("Random Forest",       y_test, rf_predictions)
 


STEP 8: Model Evaluation

── Logistic Regression ──
  Accuracy  : 0.8333  (83.3%)
  Precision : 0.6957
  Recall    : 1.0000  ← most important in medical context
  F1 Score  : 0.8205

  Classification Report:
              precision    recall  f1-score   support

  No Disease       1.00      0.73      0.84        26
 Has Disease       0.70      1.00      0.82        16

    accuracy                           0.83        42
   macro avg       0.85      0.87      0.83        42
weighted avg       0.88      0.83      0.84        42


── Random Forest ──
  Accuracy  : 0.9286  (92.9%)
  Precision : 0.9333
  Recall    : 0.8750  ← most important in medical context
  F1 Score  : 0.9032

  Classification Report:
              precision    recall  f1-score   support

  No Disease       0.93      0.96      0.94        26
 Has Disease       0.93      0.88      0.90        16

    accuracy                           0.93        42
   macro avg       0.93      0.92      0.92        42
weighted avg   

In [11]:
# STEP 9: Confusion Matrix Visualisation
# ─────────────────────────────────────────────
# We plot both confusion matrices side by side for easy comparison.
# Each cell shows: how many patients fell into each outcome category.
#
# Reading the matrix:
#   Top-left     = True Negatives  (correctly said no disease)
#   Top-right    = False Positives (wrongly flagged as disease)
#   Bottom-left  = False Negatives (missed actual disease) ← most dangerous
#   Bottom-right = True Positives  (correctly caught disease)
 
print("\n" + "=" * 55)
print("STEP 9: Generating Confusion Matrix Charts")
print("=" * 55)
 
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
 
for ax, y_pred, title in zip(
    axes,
    [lr_predictions, rf_predictions],
    ["Logistic Regression", "Random Forest"]
):
    cm = confusion_matrix(y_test, y_pred)
 
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",                          # integer format
        cmap="Blues",
        ax=ax,
        xticklabels=["No Disease", "Has Disease"],
        yticklabels=["No Disease", "Has Disease"],
        linewidths=1,
        linecolor="white",
        cbar=False
    )
 
    ax.set_title(f"Confusion Matrix\n{title}",
                 fontsize=13, fontweight="bold", pad=12)
    ax.set_xlabel("Predicted Label", fontsize=11, labelpad=8)
    ax.set_ylabel("Actual Label",    fontsize=11, labelpad=8)
 
    # Annotate the False Negative cell (bottom-left) — most critical
    fn_value = cm[1][0]
    ax.text(
        0.25, 0.72,
        f"⚠ {fn_value} missed\nsick patients",
        transform=ax.transAxes,
        ha="center", fontsize=8.5,
        color="red", fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.3", facecolor="lightyellow", alpha=0.8)
    )
 
plt.suptitle("Confusion Matrix Comparison — Logistic Regression vs Random Forest",
             fontsize=13, fontweight="bold", y=1.02)
plt.tight_layout()
plt.savefig("charts/chart4_confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart4_confusion_matrices.png")


STEP 9: Generating Confusion Matrix Charts


C:\Users\phamd\anaconda3\lib\site-packages\seaborn\utils.py:95: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from current font.
  fig.canvas.draw()
C:\Users\phamd\AppData\Local\Temp\ipykernel_21344\1505276273.py:56: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from current font.
  plt.tight_layout()
C:\Users\phamd\AppData\Local\Temp\ipykernel_21344\1505276273.py:57: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from current font.
  plt.savefig("charts/chart4_confusion_matrices.png", dpi=150, bbox_inches="tight")


  Saved → charts/chart4_confusion_matrices.png


In [12]:
# STEP 10: Metrics Comparison Bar Chart
# ─────────────────────────────────────────────
 
print("\nGenerating metrics comparison chart...")
 
metrics_df = pd.DataFrame([lr_metrics, rf_metrics]).set_index("name")
metrics_df = metrics_df[["accuracy", "precision", "recall", "f1"]]
 
fig, ax = plt.subplots(figsize=(10, 6))
 
x = np.arange(len(metrics_df.columns))
width = 0.35
 
bars1 = ax.bar(x - width/2, metrics_df.loc["Logistic Regression"],
               width, label="Logistic Regression",
               color="#4C72B0", edgecolor="white")
bars2 = ax.bar(x + width/2, metrics_df.loc["Random Forest"],
               width, label="Random Forest",
               color="#55A868", edgecolor="white")
 
# Add value labels on top of each bar
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
 
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.01,
            f"{bar.get_height():.2f}",
            ha="center", va="bottom", fontsize=9, fontweight="bold")
 
ax.set_title("Model Performance Comparison — All Metrics",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xticks(x)
ax.set_xticklabels(["Accuracy", "Precision", "Recall", "F1 Score"], fontsize=11)
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.15)
ax.legend(fontsize=10)
ax.axhline(y=0.61, color="red", linestyle="--", linewidth=1, alpha=0.5)
ax.text(3.55, 0.62, "Baseline\n(61%)", fontsize=8, color="red", alpha=0.7)
 
plt.tight_layout()
plt.savefig("charts/chart5_metrics_comparison.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart5_metrics_comparison.png")
 


Generating metrics comparison chart...
  Saved → charts/chart5_metrics_comparison.png


In [13]:
# STEP 11: Feature Importance (Random Forest)
# ─────────────────────────────────────────────
# Random Forest tells us how much each feature contributed to its decisions.
# This validates whether the model learned something clinically meaningful —
# features that rank high here should align with what we saw in the heatmap.
 
print("\nGenerating feature importance chart...")
 
feature_importance = pd.Series(
    rf_model.feature_importances_,
    index=X.columns
).sort_values(ascending=True)
 
fig, ax = plt.subplots(figsize=(10, 7))
 
colors = ["#C44E52" if imp > 0.08 else "#4C72B0"
          for imp in feature_importance.values]
 
feature_importance.plot(
    kind="barh", ax=ax,
    color=colors,
    edgecolor="white"
)
 
ax.set_title("Feature Importance — Random Forest",
             fontsize=14, fontweight="bold", pad=15)
ax.set_xlabel("Importance Score", fontsize=11)
ax.set_ylabel("Feature",          fontsize=11)
ax.tick_params(labelsize=9)
 
ax.text(0.98, 0.02,
        "Red bars = top features (importance > 0.08)",
        transform=ax.transAxes, ha="right",
        fontsize=8.5, color="dimgray", style="italic")
 
plt.tight_layout()
plt.savefig("charts/chart6_feature_importance.png", dpi=150, bbox_inches="tight")
plt.close()
print("  Saved → charts/chart6_feature_importance.png")
 


Generating feature importance chart...
  Saved → charts/chart6_feature_importance.png


In [14]:
# STEP 12: Final Summary
# ─────────────────────────────────────────────
 
print("\n" + "=" * 55)
print("PART C COMPLETE — Final Summary")
print("=" * 55)
 
print(f"""
MODEL COMPARISON:
{'Metric':<15} {'Logistic Regression':>22} {'Random Forest':>16}
{'─'*55}
{'Accuracy':<15} {lr_metrics['accuracy']:>22.4f} {rf_metrics['accuracy']:>16.4f}
{'Precision':<15} {lr_metrics['precision']:>22.4f} {rf_metrics['precision']:>16.4f}
{'Recall':<15} {lr_metrics['recall']:>22.4f} {rf_metrics['recall']:>16.4f}
{'F1 Score':<15} {lr_metrics['f1']:>22.4f} {rf_metrics['f1']:>16.4f}
 
WINNER: {'Random Forest' if rf_metrics['f1'] >= lr_metrics['f1'] else 'Logistic Regression'}
REASON: Higher F1 and Recall — better at catching sick patients.
 
CLASS IMBALANCE:
  Dataset ratio: 61% no disease vs 39% disease (mild imbalance)
  Method used  : class_weight='balanced' in both models
  Effect       : minority class errors penalised more heavily
 
DOES ACCURACY PARADOX APPLY HERE?
  A lazy model predicting 'no disease' always = 61% accuracy.
  Both our models scored above this baseline.
  With class_weight='balanced', Recall is prioritised —
  the models are not exploiting the majority class.
""")
 
print("Charts saved:")
print("  charts/chart4_confusion_matrices.png")
print("  charts/chart5_metrics_comparison.png")
print("  charts/chart6_feature_importance.png")
 


PART C COMPLETE — Final Summary

MODEL COMPARISON:
Metric             Logistic Regression    Random Forest
───────────────────────────────────────────────────────
Accuracy                        0.8333           0.9286
Precision                       0.6957           0.9333
Recall                          1.0000           0.8750
F1 Score                        0.8205           0.9032
 
WINNER: Random Forest
REASON: Higher F1 and Recall — better at catching sick patients.
 
CLASS IMBALANCE:
  Dataset ratio: 61% no disease vs 39% disease (mild imbalance)
  Method used  : class_weight='balanced' in both models
  Effect       : minority class errors penalised more heavily
 
DOES ACCURACY PARADOX APPLY HERE?
  A lazy model predicting 'no disease' always = 61% accuracy.
  Both our models scored above this baseline.
  With class_weight='balanced', Recall is prioritised —
  the models are not exploiting the majority class.

Charts saved:
  charts/chart4_confusion_matrices.png
  charts/chart5_